# Experimento 1 — Random Forest com 3 Classes (Sem Balanceamento)

## Objetivo do experimento

Após a realização dos primeiros experimentos com 4 classes (`Excelente`, `Boa`, `Atenção` e `Crítica`), foi observado um forte desbalanceamento entre as categorias, principalmente na classe `Crítica`, que possuía uma quantidade muito pequena de amostras em comparação às demais.

Esse desbalanceamento acabou impactando diretamente:

* a capacidade de aprendizado do modelo;
* o recall da classe crítica;
* a generalização;
* e o nível de overfitting.

Diante disso, foi criada uma nova estratégia de rotulagem utilizando apenas 3 classes:

* `Adequada`
* `Atenção`
* `Crítica`

A nova divisão foi definida da seguinte forma:

* score 5 → `Adequada`
* score 3 e 4 → `Atenção`
* score 0, 1 e 2 → `Crítica`

O objetivo dessa nova estrutura é:

* reduzir fragmentação entre classes;
* aumentar a quantidade de amostras das classes intermediárias;
* facilitar o aprendizado do modelo;
* reduzir overfitting;
* melhorar generalização;
* e tornar as fronteiras de decisão menos complexas.

---

## Estrutura do experimento

Neste primeiro experimento da nova abordagem, será utilizado:

* algoritmo Random Forest;
* 4 variáveis principais;
* sem aplicação de balanceamento;
* utilizando o novo rótulo com apenas 3 classificações.

As variáveis utilizadas serão:

* temperatura;
* ortofosfato;
* country;
* waterbody type.

O objetivo inicial é entender:

* como o modelo se comporta com a nova distribuição das classes;
* se houve melhora no equilíbrio das previsões;
* se o overfitting diminuiu;
* e se o modelo passou a generalizar melhor em relação aos experimentos anteriores.


## Preparação do ambiente


In [ ]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada_3.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/amostra_rotulada_3.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Adequada
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Adequada
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Adequada


In [ ]:
# preparando o ambiente para machine learning
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# definição de x e y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

In [ ]:
#train_test split

SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)


Treino: (113119, 4)
Teste: (28280, 4)


In [ ]:
# pré-processamento

categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            RandomForestClassifier(
                random_state=SEED
            )
        )
    ]
)

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi realizada a avaliação das métricas no conjunto de treino.

Essa etapa é importante porque permite comparar o comportamento do modelo entre treino e teste, ajudando na identificação de possíveis sinais de overfitting.

Por esse motivo, não basta apenas observar uma alta acurácia no treino. O ideal é que os resultados de treino e teste permaneçam relativamente próximos, indicando que o modelo conseguiu aprender padrões mais generalizáveis ao invés de apenas memorizar os dados.

Além da acurácia, também foram analisadas métricas como precision, recall e F1-score, permitindo observar como o modelo se comporta em cada classe individualmente.

In [ ]:
y_train_pred = model.predict(X_train)

In [ ]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.9003527258904339
Train Precision:
0.8985126856743727
Train Recall:
0.9003527258904339
Train F1:
0.8976804198169587
Train Confusion Matrix:
[[79417  3339    19]
 [ 7484 21728    26]
 [  103   301   702]]


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# Teste com 4 variáveis (3 classificações) - sem balanceamento
print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7441654879773691

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.81      0.87      0.84     20694
     Atenção       0.51      0.42      0.46      7309
     Crítica       0.08      0.03      0.04       277

    accuracy                           0.74     28280
   macro avg       0.47      0.44      0.45     28280
weighted avg       0.72      0.74      0.73     28280


Confusion Matrix:
[[18003  2660    31]
 [ 4227  3035    47]
 [   71   199     7]]


## Resultados Obtidos — Comparação entre 4 Classes e 3 Classes

### Modelo com 4 classes

Resultados:

* treino ≈ 88%
* teste ≈ 71%

O modelo apresentou:

* forte dificuldade nas classes intermediárias;
* baixo recall para `Atenção` `Boa` `Crítica`;
* e forte tendência em prever a classe `Excelente`.

Isso ocorreu porque:

* as fronteiras entre `Boa` e `Atenção` eram muito próximas;
* o problema ficou excessivamente fragmentado;
* e a classe `Crítica` possuía poucas amostras.

---

## Modelo com 3 classes

Resultados:

* treino ≈ 90%
* teste ≈ 74%

A simplificação para 3 classificações trouxe melhorias importantes:

* aumento da acurácia;
* melhora significativa da classe intermediária;
* redução parcial da fragmentação;
* e simplificação do problema de classificação.

A nova classe `Atenção`, formada pela união das antigas classes `Boa` e `Atenção`, passou a possuir uma quantidade muito maior de exemplos, permitindo que o modelo aprendesse melhor seus padrões.

Como consequência:

* o recall da classe intermediária aumentou significativamente;
* o modelo passou a identificar melhor padrões não totalmente adequados;
* e a generalização apresentou melhora parcial.

---

# Impacto da redução das classes

A redução da quantidade de classificações simplificou as fronteiras de decisão do modelo.

Antes, o Random Forest precisava aprender diferenças muito sutis entre:

* `Excelente`
* `Boa`
* `Atenção`

Após a nova divisão, o problema ficou mais simples:

* `Adequada`
* `Atenção`
* `Crítica`

Isso reduziu ambiguidades e tornou o aprendizado mais estável.

---

# Classe `Crítica`

Mesmo com a nova estrutura, a classe `Crítica` continuou apresentando baixo desempenho.

Isso ocorreu porque:

* a quantidade de amostras críticas permaneceu extremamente pequena;
* o dataset continua fortemente desbalanceado;
* e o modelo ainda possui poucos exemplos para aprender padrões críticos consistentes.

O recall da classe crítica permaneceu muito baixo, indicando que o modelo ainda encontra dificuldades para reconhecer corretamente amostras críticas reais.

---

# Overfitting

Os resultados mostraram que o overfitting continua presente:

* treino ≈ 90%
* teste ≈ 74%

Isso indica que o Random Forest ainda aprende os padrões do conjunto de treino de forma muito forte, perdendo parte da capacidade de generalização no conjunto de teste.

Entretanto, mesmo com o overfitting ainda elevado, o modelo com 3 classes apresentou melhor desempenho geral em comparação ao cenário anterior com 4 classificações.

---

# Conclusão

A redução para 3 classificações apresentou resultados positivos:

* melhora da acurácia;
* melhora significativa da classe intermediária;
* simplificação do problema;
* redução parcial da fragmentação;
* e melhora geral da estabilidade do modelo.

Porém, o desbalanceamento da classe `Crítica` continua sendo um dos principais desafios do projeto.

Os próximos experimentos irão avaliar:

* aplicação de balanceamento;
* impacto de novas variáveis;
* comportamento do modelo em diferentes configurações;
* e possíveis estratégias para melhorar o reconhecimento das classes minoritárias.


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# dataframe comparativo
df = pd.DataFrame({

    "Métrica": [

        "Accuracy treino",
        "Accuracy teste",
        "Overfitting",

        "Precision Excelente/Adequada",
        "Recall Excelente/Adequada",
        "F1 Excelente/Adequada",

        "Precision Boa",
        "Recall Boa",
        "F1 Boa",

        "Precision Atenção",
        "Recall Atenção",
        "F1 Atenção",

        "Precision Crítica",
        "Recall Crítica",
        "F1 Crítica"
    ],

    "4 Classificações": [

        # accuracy
        0.884,
        0.710,
        0.174,

        # excelente
        0.80,
        0.89,
        0.84,

        # boa
        0.33,
        0.24,
        0.28,

        # atenção
        0.32,
        0.20,
        0.24,

        # crítica
        0.10,
        0.05,
        0.06
    ],

    "3 Classificações": [

        # accuracy
        0.900,
        0.744,
        0.156,

        # adequada
        0.81,
        0.87,
        0.84,

        # atenção (junção Boa + Atenção)
        "-",
        "-",
        "-",

        # atenção nova
        0.51,
        0.42,
        0.46,

        # crítica
        0.08,
        0.03,
        0.04
    ]
})

# tabela interativa
fig = go.Figure(data=[go.Table(

    header=dict(
        values=list(df.columns),
        fill_color='#0F766E',
        font=dict(color='white', size=14),
        align='center'
    ),

    cells=dict(
        values=[df[col] for col in df.columns],
        fill_color='#ECFEFF',
        align='center',
        font=dict(size=13),
        height=32
    )
)])

fig.update_layout(
    title="Comparação Experimental — 4 vs 3 Classificações",
    width=1100,
    height=700
)

fig.show()